# Different boundary conditions for the Poisson equation
Governing equations:<br>
$-\Delta u = f$ in $\Omega$<br>
$u=u_L$ on $\partial_{D_L}$ <br>
$u=u_R$ on $\partial_{D_R}$ <br>
$-\partial_n u = g$ on $\partial_N$ <br>

Weak form: <br>
$a(u,v)=\int_\Omega \nabla u \cdot \nabla v dx$ <br>
$L(v)=\int_\Omega fvdx - \int_{\partial \Omega} gvds$ <br>

Here: Dirichlet on $x=0 ~ (\partial_{D_L})$ and $x=1 ~ (\partial_{D_R})$ with $u_L = 1 + 2y^2$ and $u_R = 2 + 2y^2$ and Neumann on $y=0$ and $y=1$ with $g(x, 0) = 0$ and $g(x,1)=-4$. Also we choose $f(x,y)=-6$. (If we want to use only one Dirichlet b.c. here we can choose $u=u_D$ on $\partial_D$ for $x=0$ and $x=1$ with $u_D = 1 + x^2+2y^2$).

In [2]:
from dolfinx import default_scalar_type
from dolfinx.fem import (
    Constant,
    Function,
    functionspace,
    assemble_scalar,
    dirichletbc,
    form,
    locate_dofs_geometrical,
)
from dolfinx.fem.petsc import LinearProblem
from dolfinx.mesh import create_unit_square
from dolfinx.plot import vtk_mesh

from mpi4py import MPI
from ufl import SpatialCoordinate, TestFunction, TrialFunction, dot, ds, dx, grad

import numpy as np
import pyvista

In [3]:
# Define grid
domain = create_unit_square(MPI.COMM_WORLD, 10, 10)
V = functionspace(domain, ("Lagrange", 1))

# Define test, trial function, lhs
u = TrialFunction(V)
v = TestFunction(V)
a = dot(grad(u), grad(v))*dx

In [4]:
# Define Dirichlet boundary conditions (if we consider only one boundary fct: u_D = 1+x^2+2y^2)

def u_exact(x):
    return 1 + x[0]**2 + 2*x[1]**2

def on_Dirichlet_bd(x):
    return np.logical_or(np.isclose(x[0], 0), np.isclose(x[0], 1))

dofs_D = locate_dofs_geometrical(V, on_Dirichlet_bd)
uD = Function(V)
uD.interpolate(u_exact)
bc_D = dirichletbc(uD, dofs_D)


In [5]:
# Define two Dirichlet b.c.
dofs_L = locate_dofs_geometrical(V, lambda x: np.isclose(x[0], 0))
u_L = Function(V)
u_L.interpolate(lambda x: 1+2*x[1]**2)
bc_L = dirichletbc(u_L, dofs_L)

dofs_R = locate_dofs_geometrical(V, lambda x: np.isclose(x[0], 1))
u_R = Function(V)
u_R.interpolate(lambda x: 2+2*x[1]**2)
bc_R = dirichletbc(u_R, dofs_R)

bcs = [bc_L, bc_R]

In [6]:
# Define Neumann b.c.
x = SpatialCoordinate(domain)
g = -4*x[1]
f = Constant(domain, default_scalar_type(-6))

L = f*v*dx - g*v*ds

In [7]:
# Solve linear problem
problem = LinearProblem(a, L, bcs = bcs, 
                        petsc_options={"ksp_type": "preonly", "pc_type": "lu"},petsc_options_prefix="neumann_dirichlet_",
)
uh = problem.solve()

In [8]:
# Error analysis
V2 = functionspace(domain, ("Lagrange", 2))
u_ex = Function(V2)
u_ex.interpolate(u_exact)

error_L2 = assemble_scalar(form((uh - u_ex) ** 2 * dx))
error_L2 = np.sqrt(MPI.COMM_WORLD.allreduce(error_L2, op=MPI.SUM))

uh_vertex_vals = uh.x.array
u_ex_vertex = Function(V)
u_ex_vertex.interpolate(u_exact)
u_ex_vertex_vals = u_ex_vertex.x.array
error_max = np.max(np.abs(uh_vertex_vals - u_ex_vertex_vals))
error_max = MPI.COMM_WORLD.allreduce(error_max, op=MPI.MAX)

print(f"L2-Error: {error_L2:.2e}")
print(f"Max Error at nodes: {error_max:.2e}")

L2-Error: 5.27e-03
Max Error at nodes: 2.44e-15


In [10]:
# Plot
from visualization_fct import plotScalarFunction
plotScalarFunction(V, uh, warped=True)

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…